In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [72]:
df=pd.read_csv(r"C:\Users\RAJDEEP\Downloads\cardekho_imputated.csv")

In [73]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [74]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [75]:
df.drop("car_name",axis=1,inplace=True)
df.drop('brand' ,axis=1,inplace=True)


In [76]:
df['model'].unique()


<ArrowStringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [77]:
from sklearn.model_selection import train_test_split
x=df.drop(['selling_price'],axis=1)
y=df['selling_price']

In [78]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
x['model']=le.fit_transform(x['model'])

In [55]:
num_features=[feature for feature in df.columns if df[feature].dtype!='O']
print("no of numerical feature", len(num_features))
cat_features=[feature for feature in df.columns if df[feature].dtype=='O']
print("no of categorical feature", len(cat_features))
dis_features=[feature for feature in df.columns if len(df[feature].unique())<=25]
print("no of discrete feature", len(dis_features))
#continuoues featuere
cont_features=[feature for feature in num_features if feature not in dis_features]
print("no of continuous feature", len(cont_features))


no of numerical feature 12
no of categorical feature 0
no of discrete feature 5
no of continuous feature 7


In [57]:
len(df['model'].unique())

120

In [80]:
num_features=x.select_dtypes(exclude='object').columns
onehot_columns=['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder(drop='first')
preprocess=ColumnTransformer(
    [
    ('OneHotEncoder',oh_transformer,num_features),
    ('StandardScaler',numeric_transformer,num_features)
    ],remainder='passthrough'
)

In [ ]:
x=preprocess.fit_transform(x)


In [66]:
x_t,x_test,y_t,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
x_t.shape


(10787, 11)

In [68]:
print(type(x_t))
print(x_t[0])

<class 'numpy.ndarray'>
[-0.2824368601094044 0.29742112897872114 -0.6758070959398955
 -0.2831725285393187 0.10521104622155251 0.02291783356564712
 -0.27665383749659156 -0.4030224142720608 'Dealer' 'Diesel' 'Manual']


In [46]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge,Lasso
from sklearn.metrics import accuracy_score,r2_score,recall_score,precision_score,f1_score,roc_auc_score,classification_report,confusion_matrix,mean_absolute_error,mean_squared_error,root_mean_squared_error

In [47]:
def evaluate_model(true,predicted):
    mse=mean_squared_error(true,predicted)
    mae=mean_absolute_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(true,predicted)
    return mae,rmse,r2_square


In [67]:
models={
    'Randon Forest':RandomForestRegressor(),
    'Decision Tree':DecisionTreeRegressor(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'KNeighbors Regressor':KNeighborsRegressor(),

}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_t,y_t)
    y_train_pred=model.predict(x_t)
    y_test_pred=model.predict(x_test)
    model_train_mae,model_train_rmse,model_train_r2=evaluate_model(y_t,y_train_pred)
    model_test_mae,model_test_rmse,model_test_r2=evaluate_model(y_test,y_test_pred)
    print(list(models.keys())[i])
    print("root mean squares".format(model_train_rmse))
    print("absolute error".format(model_train_mae))
    print("r2 score".format(model_train_r2))
    #test
    print("accuracy".format(model_test_rmse))
    print("accuracy".format(model_test_mae))
    print("accuracy".format(model_test_r2))

ValueError: could not convert string to float: 'Dealer'

In [79]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(r"C:\Users\RAJDEEP\Downloads\cardekho_imputated.csv")

# Drop unwanted columns
df.drop(["car_name", "brand"], axis=1, inplace=True)

# Features and Target
X = df.drop("selling_price", axis=1)
y = df["selling_price"]

# Label Encode model column
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X["model"] = le.fit_transform(X["model"])

# Numerical and Categorical columns
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = ["seller_type", "fuel_type", "transmission_type"]

# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first", handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Apply preprocessing
X_train = preprocess.fit_transform(X_train)
X_test = preprocess.transform(X_test)

# Models
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge, Lasso

models = {
    "Random Forest": RandomForestRegressor(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "KNeighbors": KNeighborsRegressor()
}

# Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# Train and Evaluate
for name, model in models.items():

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mae, train_rmse, train_r2 = evaluate_model(y_train, train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, test_pred)

    print("=" * 60)
    print(name)

    print("Train MAE :", round(train_mae, 2))
    print("Train RMSE:", round(train_rmse, 2))
    print("Train R2  :", round(train_r2, 4))

    print("Test MAE  :", round(test_mae, 2))
    print("Test RMSE :", round(test_rmse, 2))
    print("Test R2   :", round(test_r2, 4))

Random Forest
Train MAE : 36150.32
Train RMSE: 129840.66
Train R2  : 0.9794
Test MAE  : 102462.04
Test RMSE : 248820.55
Test R2   : 0.9177
Decision Tree
Train MAE : 0.0
Train RMSE: 0.0
Train R2  : 1.0
Test MAE  : 139297.12
Test RMSE : 410365.88
Test R2   : 0.7761
Lasso
Train MAE : 268441.82
Train RMSE: 559304.06
Train R2  : 0.6183
Test MAE  : 281388.6
Test RMSE : 507601.31
Test R2   : 0.6574
Ridge
Train MAE : 268398.51
Train RMSE: 559304.89
Train R2  : 0.6183
Test MAE  : 281337.05
Test RMSE : 507584.93
Test R2   : 0.6574
KNeighbors
Train MAE : 99251.77
Train RMSE: 346452.48
Train R2  : 0.8535
Test MAE  : 127864.73
Test RMSE : 316793.51
Test R2   : 0.8666
